In [ ]:
# Q8: TOP 10 NGÀY CÓ TẦN SUẤT KÍCH HOẠT LOW PRESSURE SWITCH (LPS) CAO NHẤT
q8 = spark.sql('''
    SELECT
        date,
        COUNT(*) AS tong_so_ban_ghi,
        SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) AS so_lan_lps_kich_hoat,
        ROUND(
            SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS ty_le_lps_kich_hoat_pct
    FROM sensor
    GROUP BY date
    HAVING SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) > 0
    ORDER BY so_lan_lps_kich_hoat DESC
    LIMIT 10
''')

# [GIỮ NGUYÊN] In kế hoạch thực thi chi tiết cho báo cáo
print("\n========== EXECUTION PLAN ==========")
q8.explain(True)

# [GIỮ NGUYÊN] Chuyển đổi sang Pandas sau khi giải thích xong
print("\n--- TOP 10 NGÀY CÓ TẦN SUẤT KÍCH HOẠT LPS CAO NHẤT ---")
df_q8 = q8.toPandas()

# Phần hiển thị: Giữ nguyên logic của bạn nhưng thêm bẫy kiểm tra hiển thị
if df_q8.empty:
    print("Mẹo: Bảng kết quả trả về 0 dòng. Điều này có nghĩa là trong tập dữ liệu sensor hiện tại, không có bản ghi nào mà LPS có giá trị bằng 1.")
else:
    try:
        display(df_q8)
    except NameError:
        # Ép in trực tiếp ra console nếu hàm display() của môi trường không hoạt động
        print(df_q8.to_string(index=False))

In [ ]:
# Q9: PHÂN TÍCH CÁC PHIÊN VẬN HÀNH LIÊN TỤC DÀI NHẤT
q9 = spark.sql('''
WITH session_edges AS (
    SELECT
        timestamp,
        COMP,
        Motor_current,
        LAG(COMP,1,1) OVER(ORDER BY timestamp) AS prev_comp FROM sensor),
session_blocks AS (
    SELECT
        timestamp,
        COMP,
        Motor_current,
        SUM(
            CASE
                WHEN COMP = 0 AND prev_comp = 1 THEN 1
                ELSE 0 END
        ) OVER(ORDER BY timestamp) AS session_id
    FROM session_edges)
SELECT
    session_id,
    MIN(timestamp) AS session_start,
    MAX(timestamp) AS session_end,
    COUNT(*) AS duration_seconds,
    ROUND(AVG(Motor_current),2) AS avg_current_in_session
FROM session_blocks
WHERE COMP = 0
GROUP BY session_id
ORDER BY duration_seconds DESC
LIMIT 10''')
print("\n========== EXECUTION PLAN ==========")
q9.explain(True)
print("\n--- TOP 10 PHIÊN VẬN HÀNH LIÊN TỤC DÀI NHẤT ---")
df_q9 = q9.toPandas()
try:
    display(df_q9)
except NameError:
    print(df_q9.to_string(index=False))

In [ ]:
# Q10: TOP 10 NGÀY CÓ TẦN SUẤT ĐÓNG/NGẮT CAO NHẤT
q10 = spark.sql('''
WITH state_changes AS (
    SELECT
        date,
        timestamp,
        COMP,
        LAG(COMP,1,COMP) OVER(ORDER BY timestamp) AS prev_comp
    FROM sensor
)
SELECT
    date,
    COUNT(*) AS so_lan_dong_ngat
FROM state_changes
WHERE COMP <> prev_comp
GROUP BY date
ORDER BY so_lan_dong_ngat DESC
LIMIT 10
''')
print("\n========== EXECUTION PLAN ==========")
q10.explain(True)
print("\n--- TOP 10 NGÀY CÓ TẦN SUẤT ĐÓNG/NGẮT CAO NHẤT ---")
df_q10 = q10.toPandas()
try:
    display(df_q10)
except NameError:
    print(df_q10.to_string(index=False))